# Knowledge layer: knowledge base + knowledge graph
**KB**: clean → quality gate → hash-idempotent ingest → chunk → incremental embed → BM25 / dense / RRF search, grouped by bug, time-cut, own run excluded. **KG**: extract → purify (counted rules) → load (hash diff) → openCypher projection; domain / bug / expert views and leak-guarded personalised PageRank. Production runs both in one Postgres (pgvector + Apache AGE, see schema.sql).

> All data in this notebook is **synthetic** (generated below). The real system runs on private validation data; the code paths are the same, the numbers in the README are from the real data.

In [ ]:
!pip -q install numpy sentence-transformers

In [ ]:
%%writefile synthetic.py
"""Synthetic failure generator for the demo notebooks (no real data is published).
Each bug has a signature (2-3 log lines), a domain, an owner and a test family; runs of that bug emit the signature
mixed with background noise, sometimes miss a line, and 'sibling' bugs share most of their signature, which is the
hard case in the real system. Not-a-bug runs are infrastructure or test-content problems."""
import random
from datetime import datetime, timedelta

DOMAINS = ["power", "memory", "reset", "virtualization", "media", "compute", "fabric", "firmware"]
COMPONENTS = ["gt", "dma", "pcie", "fw", "mem", "sched", "link", "irq", "pmu", "vf"]
SYMPTOMS = ["engine reset", "timeout waiting for fence", "page fault at <HEX>", "device lost", "link training failed",
            "ECC uncorrectable", "watchdog expired", "CRC mismatch", "thermal trip", "context banned"]
NOISE = ["test step <N> started", "allocating <N> buffers", "INFO: iteration <N> passed", "loading module", "heartbeat ok",
         "perf counter sample <N>", "config applied", "mounting share", "INFO: retrying connection"]
GENERIC = ["FAIL: test step <N> returned non-zero", "ERROR: timeout waiting for test harness", "FAIL: assertion in test script"]
INFRA = ["ERROR: node unreachable", "ERROR: container image pull failed", "FAIL: missing test binary", "ERROR: disk quota exceeded"]


def make_world(n_bugs=40, n_runs=1200, p_none=.18, seed=0):
    rng = random.Random(seed)
    t0 = datetime(2026, 1, 1)
    bugs = []
    for i in range(n_bugs):
        sib = bugs[rng.randrange(len(bugs))] if bugs and rng.random() < .35 else None
        comp = sib["component"] if sib else rng.choice(COMPONENTS)
        sig = list(sib["signature"][:2]) if sib else [f"{comp}: {rng.choice(SYMPTOMS)}", f"{comp}: {rng.choice(SYMPTOMS)} on <PCI>"]
        sig.append(f"{comp}: first fault code {rng.randrange(16, 255):#x}")
        bugs.append({"id": f"BUG-{i:03d}", "domain": sib["domain"] if sib else rng.choice(DOMAINS), "component": comp,
                     "owner": f"owner{rng.randrange(8)}", "family": f"T{rng.randrange(12)}", "signature": sig,
                     "created": t0 + timedelta(days=rng.randrange(0, 200)),
                     "title": f"[{comp}] {sig[0].split(': ', 1)[1]}", "root_cause": f"{comp} {sig[-1].split(': ', 1)[1]}"})
    w = [1 / (i + 1) ** .8 for i in range(n_bugs)]          # long tail: a few bugs own most failures
    runs = []
    for j in range(n_runs):
        start = t0 + timedelta(days=rng.uniform(0, 260))
        if rng.random() < p_none:
            lines, label, b = ([rng.choice(INFRA)] if rng.random() < .6 else [rng.choice(GENERIC)]), "NONE", None
        else:
            b = rng.choices(bugs, w)[0]
            if b["created"] > start + timedelta(days=30):
                start = b["created"] + timedelta(days=rng.uniform(0, 30))
            lines, label = [s for s in b["signature"] if rng.random() > .3] or b["signature"][:1], b["id"]
            if rng.random() < .25:
                lines.append(rng.choice(GENERIC + INFRA))      # product bugs also trip generic/infra-looking lines
        lines += rng.sample(NOISE, 4)
        rng.shuffle(lines)
        runs.append({"run": f"RUN-{j:05d}", "label": label, "start": start, "lines": lines,
                     "domain": b["domain"] if b else rng.choice(DOMAINS), "owner": b["owner"] if b else f"owner{rng.randrange(8)}",
                     "family": b["family"] if b and rng.random() < .7 else f"T{rng.randrange(12)}"})
    return bugs, runs


def split(runs, bugs, seed=0):
    """Grouped by bug: 30% of bugs are test-only, so the test split measures bugs never seen in training."""
    rng = random.Random(seed)
    ids = [b["id"] for b in bugs]
    rng.shuffle(ids)
    test_bugs = set(ids[: int(.3 * len(ids))])
    for r in runs:
        r["split"] = "test" if (r["label"] in test_bugs or (r["label"] == "NONE" and rng.random() < .3)) else "train"
        # frequent bugs also appear in test (seen-bug slice)
        if r["label"] not in test_bugs and r["label"] != "NONE" and rng.random() < .2:
            r["split"] = "test"
    return runs

In [ ]:
%%writefile knowledge.py
"""Knowledge layer: structured bug cards + hybrid search + a small knowledge graph.

1. Bug cards (LLM, once per bug, cached): free-text bug reports are long and noisy (console pastes, HTML).
   An LLM turns each into a card: component, symptom, first fault, match phrases (verbatim log lines),
   distinguishing detail vs sibling bugs, workaround. Cards are validated: a match phrase that never occurs in
   any linked run's evidence is dropped.
2. Hybrid search: BM25 (exact log vocabulary) + dense embeddings (paraphrase) fused with reciprocal-rank fusion.
   Used by the agent as a tool and to widen the ranker's candidate pool.
3. Knowledge graph: nodes = bugs, components, tests, owners, root causes; edges = links, supersedes, shares
   component, co-occurs. Queried by the agent for neighbourhoods ("siblings of this bug").
Everything respects the run's cutoff: bugs filed after the run, and the run's own link, are invisible.
"""
from __future__ import annotations

import json
import math
import re
from collections import Counter, defaultdict

CARD_PROMPT = """You turn one bug report into a structured card for a log-matching system.
Return JSON with keys: component, symptom, first_fault, match_phrases (3-8 verbatim log lines that identify THIS bug),
distinguishing (what separates it from similar bugs), workaround.
Only copy phrases that appear verbatim in the report. Do not invent log lines.

Report:
{report}"""


def make_card(report: str, llm) -> dict:
    card = json.loads(llm(CARD_PROMPT.format(report=report[:12000])))
    card["match_phrases"] = [p for p in card.get("match_phrases", []) if p and p in report]  # verbatim only
    return card


def validate_card(card: dict, linked_evidence: list[str]) -> dict:
    """Drop phrases that never hit a run linked to this bug (they would only add noise to retrieval)."""
    text = "\n".join(linked_evidence).lower()
    card["match_phrases"] = [p for p in card["match_phrases"] if p.lower() in text] or card["match_phrases"][:2]
    return card


# ------------------------------------------------------------------ hybrid search
def _tok(s):
    return re.findall(r"[a-z0-9_]{2,}", s.lower())


class BM25:
    def __init__(self, docs: list[str], k1=1.4, b=.75):
        self.d = [_tok(x) for x in docs]
        self.avg = sum(map(len, self.d)) / max(1, len(self.d))
        self.df = Counter(t for doc in self.d for t in set(doc))
        self.k1, self.b, self.N = k1, b, len(docs)

    def scores(self, q: str):
        qt = _tok(q)
        out = []
        for doc in self.d:
            tf = Counter(doc)
            s = 0.0
            for t in qt:
                if t in tf:
                    idf = math.log(1 + (self.N - self.df[t] + .5) / (self.df[t] + .5))
                    s += idf * tf[t] * (self.k1 + 1) / (tf[t] + self.k1 * (1 - self.b + self.b * len(doc) / self.avg))
            out.append(s)
        return out


def rrf(*rankings: list[int], k: int = 60) -> list[int]:
    """Reciprocal-rank fusion of several ranked id lists."""
    score = defaultdict(float)
    for r in rankings:
        for pos, i in enumerate(r):
            score[i] += 1 / (k + pos + 1)
    return sorted(score, key=lambda i: -score[i])


class KB:
    def __init__(self, cards: list[dict], embed=None):
        self.cards = cards
        self.docs = [" ".join([c["title"], c.get("symptom", ""), c.get("first_fault", ""), *c.get("match_phrases", [])]) for c in cards]
        self.bm25 = BM25(self.docs)
        self.embed = embed
        self.E = embed(self.docs) if embed else None

    def search(self, query: str, k: int = 10, cutoff=None, hide: set | None = None) -> list[dict]:
        s = self.bm25.scores(query)
        bm = sorted(range(len(self.docs)), key=lambda i: -s[i])
        lists = [bm]
        if self.embed is not None:
            q = self.embed([query])[0]
            sims = (self.E @ q) / ((self.E ** 2).sum(1) ** .5 * (q @ q) ** .5 + 1e-9)
            lists.append(list(sims.argsort()[::-1]))
        out = []
        for i in rrf(*lists):
            c = self.cards[i]
            if (cutoff and c["created"] > cutoff) or (hide and c["id"] in hide):
                continue                                   # time cut + label hiding: no leakage
            out.append(c)
            if len(out) == k:
                break
        return out


# ------------------------------------------------------------------ knowledge graph
class Graph:
    def __init__(self):
        self.adj = defaultdict(set)

    def add(self, a, rel, b):
        self.adj[a].add((rel, b))
        self.adj[b].add((f"~{rel}", a))

    def neighbourhood(self, node, depth=2):
        seen, frontier = {node}, [node]
        for _ in range(depth):
            nxt = []
            for n in frontier:
                for _, m in self.adj[n]:
                    if m not in seen:
                        seen.add(m)
                        nxt.append(m)
            frontier = nxt
        return seen - {node}

    def siblings(self, bug):
        comps = {b for r, b in self.adj[bug] if r == "component"}
        return {x for c in comps for r, x in self.adj[c] if r == "~component" and x != bug}

In [ ]:
%%writefile kb_store.py
"""Knowledge base: everything people wrote about bugs (reports, discussion threads, run comments), searchable.

    sources ─► clean (normalise, quality ok|short|noise, mask ids/URLs, extract entities)
            ─► ingest (content-hash idempotent upsert; noise is stored but never chunked)
            ─► chunk (<= 1100 chars, 120 overlap)
            ─► embed (incremental, only quality-ok chunks; halfvec(1024) + HNSW in Postgres/pgvector)
            ─► search: bm25 | dense | fuse (RRF k=60), grouped by bug, time-cut, own-run excluded

Production storage is Postgres + pgvector (schema.sql); this module keeps the same logic in memory so the
notebook runs anywhere. Daily refresh re-ingests only new/edited docs and embeds only new chunks (~20 s).
"""
from __future__ import annotations

import hashlib
import re
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime

import numpy as np

from knowledge import BM25, rrf

# ---------------------------------------------------------------- clean
_URL = re.compile(r"https?://\S+")
_ID = re.compile(r"\b(BUG|RUN)-\d+\b")
ENTITY = {
    "hex": re.compile(r"\b0x[0-9a-f]{4,}\b", re.I),
    "path": re.compile(r"(?:/[\w.-]+){2,}"),
    "symbol": re.compile(r"\b[a-z]+_[a-z0-9_]{3,}\b"),
    "ticket": re.compile(r"\b[A-Z]{2,}-\d{2,}\b"),
    "logline": re.compile(r"^\s*\w+: .*(?:error|fail|fault|timeout|reset|lost).*$", re.I | re.M),
}


@dataclass
class Doc:
    kind: str                 # report | thread_comment | run_comment
    bug: str | None
    run: str | None
    ts: datetime
    text: str
    meta: dict = field(default_factory=dict)
    quality: str = "ok"
    entities: dict = field(default_factory=dict)

    @property
    def key(self):
        return hashlib.sha1(f"{self.kind}|{self.bug}|{self.run}|{self.ts}".encode()).hexdigest()

    @property
    def hash(self):
        return hashlib.sha1(self.text.encode()).hexdigest()


def clean(d: Doc) -> Doc:
    t = re.sub(r"<[^>]+>", " ", d.text)                       # html
    ents = {k: sorted(set(rx.findall(t)))[:50] for k, rx in ENTITY.items()}
    t = _URL.sub("<URL>", t)
    t = _ID.sub(lambda m: f"{m.group(1)}-<ID>", t)            # ids masked: queries can't match by number
    t = re.sub(r"\s+", " ", t).strip()
    words = len(t.split())
    noise = re.fullmatch(r"(?i)(\+1|thanks|ok|done|same here|any update\??|ping)\W*", t) is not None
    d.text, d.entities = t, ents
    d.quality = "noise" if noise else "short" if words < 8 else "ok"
    return d


def chunks(text: str, size: int = 1100, overlap: int = 120) -> list[str]:
    out, i = [], 0
    while i < len(text):
        j = min(len(text), i + size)
        if j < len(text):                                      # cut on a sentence/space boundary when possible
            k = max(text.rfind(". ", i, j), text.rfind(" ", i, j))
            j = k + 1 if k > i + size // 2 else j
        out.append(text[i:j].strip())
        if j == len(text):
            break
        i = j - overlap
    return out


# ---------------------------------------------------------------- store
class KnowledgeBase:
    def __init__(self, embed=None, dim: int = 1024):
        self.docs: dict[str, Doc] = {}
        self.chunk_text, self.chunk_doc, self.vec = [], [], {}
        self.embed, self.dim = embed, dim
        self._bm25 = None

    def ingest(self, docs: list[Doc]) -> dict:
        rep = defaultdict(int)
        for d in map(clean, docs):
            old = self.docs.get(d.key)
            if old and old.hash == d.hash:
                rep["unchanged"] += 1
                continue
            rep["edited" if old else "new"] += 1
            if old:                                            # drop the old chunks of an edited doc
                keep = [i for i, k in enumerate(self.chunk_doc) if k != d.key]
                self.chunk_text = [self.chunk_text[i] for i in keep]
                self.chunk_doc = [self.chunk_doc[i] for i in keep]
                self.vec = {}
            self.docs[d.key] = d
            if d.quality == "noise":
                rep["noise"] += 1
                continue
            for c in chunks(d.text):
                self.chunk_text.append(c)
                self.chunk_doc.append(d.key)
        self._bm25 = BM25(self.chunk_text) if self.chunk_text else None
        return dict(rep)

    def embed_new(self, batch: int = 64) -> int:
        """Incremental: only quality-ok chunks without a vector. Returns how many were embedded."""
        if self.embed is None:
            return 0
        todo = [i for i, k in enumerate(self.chunk_doc) if i not in self.vec and self.docs[k].quality == "ok"]
        for s in range(0, len(todo), batch):
            ids = todo[s: s + batch]
            for i, v in zip(ids, self.embed([self.chunk_text[i] for i in ids])):
                self.vec[i] = np.asarray(v, dtype=np.float16)  # halfvec: half the space, same recall
        return len(todo)

    # ------------------------------------------------------------- search
    def _visible(self, i, cutoff, exclude_run):
        d = self.docs[self.chunk_doc[i]]
        if exclude_run is not None and d.run == exclude_run:  # never read the run's own comments (leak)
            return False
        return cutoff is None or d.ts <= cutoff

    def search(self, query: str, exclude_run: str | None, k: int = 10, mode: str = "fuse", cutoff=None):
        """exclude_run is REQUIRED (pass None explicitly only for non-evaluation use). Groups chunks by bug.
        Degrades to BM25 if the embedder is down."""
        q = _ID.sub(lambda m: f"{m.group(1)}-<ID>", query)
        vis = [i for i in range(len(self.chunk_text)) if self._visible(i, cutoff, exclude_run)]
        if not vis or self._bm25 is None:
            return []
        sp = self._bm25.scores(q)
        sparse = sorted(vis, key=lambda i: -sp[i])[:200]
        ranking = sparse
        if mode in ("dense", "fuse") and self.embed is not None and self.vec:
            try:
                qv = np.asarray(self.embed([q])[0], dtype=np.float32)
                dv = [i for i in vis if i in self.vec]
                dense = sorted(dv, key=lambda i: -float(self.vec[i].astype(np.float32) @ qv))[:200]
                ranking = dense if mode == "dense" else rrf(sparse, dense)
            except Exception:                                   # embed server down -> BM25
                ranking = sparse
        out, seen = [], set()
        for i in ranking:
            b = self.docs[self.chunk_doc[i]].bug
            if b and b not in seen:
                seen.add(b)
                out.append({"bug": b, "chunk": self.chunk_text[i][:300], "kind": self.docs[self.chunk_doc[i]].kind})
            if len(out) >= k:
                break
        return out

    def thread(self, bug: str, cutoff=None, cap: int = 80) -> list[str]:
        ds = sorted((d for d in self.docs.values() if d.bug == bug and d.quality != "noise" and
                     (cutoff is None or d.ts <= cutoff)), key=lambda d: d.ts)
        return [f"[{d.kind} {d.ts:%Y-%m-%d}] {d.text}" for d in ds[:cap]]

    def by_entity(self, kind: str, value: str) -> set[str]:
        return {d.bug for d in self.docs.values() if value in d.entities.get(kind, []) and d.bug}

    def stats(self):
        q = defaultdict(int)
        for d in self.docs.values():
            q[d.quality] += 1
        return {"docs": len(self.docs), "chunks": len(self.chunk_text), "vectors": len(self.vec), **q}

In [ ]:
%%writefile kg.py
"""Knowledge graph: who/what/where around every bug, built deterministically from labelled tracker data (no LLM).

Layers (one Postgres): documents (kb_store) | curated graph tables kg.node / kg.edge (source of truth, hash-diffed,
provenance + confidence per edge) | Apache AGE openCypher projection rebuilt from kg.* on every run.
GraphRAG was considered and rejected: its indexing needs an LLM per document, while here nodes and edges come
from structured, human-labelled data. Its useful idea (per-domain communities) is kept as `domain_view`.

    extract ─► purify (every rule counts into a report) ─► load (node hash diff) ─► project (Cypher, UNWIND batches)
"""
from __future__ import annotations

import hashlib
import json
from collections import Counter, defaultdict
from dataclasses import dataclass

LABELS = {"Run", "Bug", "Test", "Domain", "Owner", "Cycle", "Config", "Host", "Job", "Entity"}
EDGES = {  # type: (from, to, provenance, default confidence)
    "LINKED_TO": ("Run", "Bug", "tracker_link", 1.0),      # 0.7 when the run did not fail
    "OF_TEST": ("Run", "Test", "enrich", 1.0),
    "IN_DOMAIN": ("Test", "Domain", "test_registry", 1.0),
    "OWNED_BY": ("Test", "Owner", "test_registry", 1.0),
    "IN_CYCLE": ("Run", "Cycle", "enrich", 1.0),
    "ON_CONFIG": ("Run", "Config", "enrich", 1.0),
    "RAN_ON": ("Run", "Host", "enrich", 1.0),
    "BUILT_BY": ("Run", "Job", "enrich", 1.0),
    "BUG_IN_DOMAIN": ("Bug", "Domain", "inferred_majority", None),   # conf = share of the bug's runs in the domain
    "MENTIONS": ("Bug", "Entity", "text_mention", .6),
    "RELATED_TO": ("Bug", "Bug", "text_mention", .5),
}


@dataclass(frozen=True)
class Edge:
    src: str
    rel: str
    dst: str
    prov: str
    conf: float


def nid(label, key):
    return f"{label}:{key}"


# ---------------------------------------------------------------- extract (no judgement)
def extract(runs: list[dict], bugs: list[dict], kb=None) -> tuple[dict, list[Edge]]:
    nodes, edges = {}, []

    def node(label, key, **p):
        n = nid(label, key)
        nodes.setdefault(n, {"label": label, "key": key, **p})
        return n

    for b in bugs:
        bn = node("Bug", b["id"], title=b.get("title", ""), created=str(b["created"]))
        for other in b.get("mentions_bugs", []):
            edges.append(Edge(bn, "RELATED_TO", nid("Bug", other), "text_mention", .5))
        if kb is not None:
            ents = [e for d in kb.docs.values() if d.bug == b["id"] for k in ("symbol", "hex", "ticket") for e in d.entities.get(k, [])]
        else:
            ents = b.get("entities", [])
        for e in ents:
            edges.append(Edge(bn, "MENTIONS", node("Entity", e), "text_mention", .6))
    for r in runs:
        rn = node("Run", r["run"], start=str(r["start"]))
        tn = node("Test", r.get("test", r["family"]))
        edges += [Edge(rn, "OF_TEST", tn, "enrich", 1.0),
                  Edge(tn, "IN_DOMAIN", node("Domain", r["domain"]), "test_registry", 1.0),
                  Edge(tn, "OWNED_BY", node("Owner", r["owner"]), "test_registry", 1.0)]
        for k, rel, lab in (("host", "RAN_ON", "Host"), ("config", "ON_CONFIG", "Config"), ("cycle", "IN_CYCLE", "Cycle")):
            if r.get(k):
                edges.append(Edge(rn, rel, node(lab, r[k]), "enrich", 1.0))
        if r["label"] != "NONE":
            edges.append(Edge(rn, "LINKED_TO", nid("Bug", r["label"]), "tracker_link", 1.0 if r.get("failed", True) else .7))
    return nodes, edges


# ---------------------------------------------------------------- purify
def purify(nodes: dict, edges: list[Edge], hub_min: int = 8, hub_share: float = .05):
    rep = Counter()
    bugs = {n for n, v in nodes.items() if v["label"] == "Bug"}
    out = []
    for e in dict.fromkeys(edges):                                     # dedupe, keep order
        if e.src == e.dst:
            rep["self_loop"] += 1; continue
        if e.rel == "RELATED_TO" and e.dst not in bugs:               # numbers in text are often run/test ids
            rep["dangling_related"] += 1; continue
        if e.rel == "LINKED_TO" and e.dst not in bugs:                # a human link to an unknown bug: keep a stub
            nodes[e.dst] = {"label": "Bug", "key": e.dst.split(":", 1)[1], "stub": True}; bugs.add(e.dst); rep["stub"] += 1
        out.append(e)
    rep["duplicate"] = len(edges) - len(dict.fromkeys(edges))
    # hub filter: an entity in 1 bug connects nothing (stays in the KB); one in > max(8, 5% of bugs) is boilerplate
    ent_bugs = defaultdict(set)
    for e in out:
        if e.rel == "MENTIONS":
            ent_bugs[e.dst].add(e.src)
    cap = max(hub_min, hub_share * len(bugs))
    drop = {x for x, s in ent_bugs.items() if len(s) <= 1 or len(s) > cap}
    rep["singleton_entity"] = sum(len(ent_bugs[x]) <= 1 for x in drop)
    rep["hub_entity"] = len(drop) - rep["singleton_entity"]
    out = [e for e in out if e.dst not in drop]
    for x in drop:
        nodes.pop(x, None)
    # derived: bug domain by majority of its linked runs (confidence = share), test of each run
    run_test = {e.src: e.dst for e in out if e.rel == "OF_TEST"}
    test_dom = {e.src: e.dst for e in out if e.rel == "IN_DOMAIN"}
    votes = defaultdict(Counter)
    for e in out:
        if e.rel == "LINKED_TO" and e.src in run_test and run_test[e.src] in test_dom:
            votes[e.dst][test_dom[run_test[e.src]]] += 1
    for b, c in votes.items():
        d, n = c.most_common(1)[0]
        out.append(Edge(b, "BUG_IN_DOMAIN", d, "inferred_majority", round(n / sum(c.values()), 2)))
        rep["mixed_domain_bug"] += len(c) > 1
    return nodes, out, dict(rep)


# ---------------------------------------------------------------- load (hash diff) and project (openCypher)
def load(store: dict, nodes: dict, edges: list[Edge]) -> dict:
    h = {n: hashlib.sha1(json.dumps(v, sort_keys=True, default=str).encode()).hexdigest() for n, v in nodes.items()}
    old = store.get("hash", {})
    diff = {"new": len(h.keys() - old.keys()), "removed": len(old.keys() - h.keys()),
            "changed": sum(1 for n in h.keys() & old.keys() if h[n] != old[n])}
    store.update(hash=h, nodes=nodes, edges=edges)                    # edges are replaced wholesale
    return diff


def project_cypher(nodes: dict, edges: list[Edge], graph: str = "hgraph", batch: int = 500) -> list[str]:
    """Statements for Apache AGE: drop+create the graph, one UNWIND per label / relation batch. AGE has no
    `SET n += map`, so each label carries a fixed property list."""
    q = [f"SELECT drop_graph('{graph}', true);", f"SELECT create_graph('{graph}');"]
    by_label = defaultdict(list)
    for n, v in nodes.items():
        by_label[v["label"]].append({"id": n, "key": v["key"]})
    for lab, rows in by_label.items():
        for i in range(0, len(rows), batch):
            lit = "[" + ",".join("{id:%s,key:%s}" % (json.dumps(r["id"]), json.dumps(str(r["key"]))) for r in rows[i:i + batch]) + "]"
            q.append(f"SELECT * FROM cypher('{graph}', $$ UNWIND {lit} AS r CREATE (:{lab} {{id:r.id, key:r.key}}) $$) AS (a agtype);")
    by_rel = defaultdict(list)
    for e in edges:
        by_rel[e.rel].append(e)
    for rel, es in by_rel.items():
        for i in range(0, len(es), batch):
            lit = "[" + ",".join("{s:%s,d:%s,p:%s,c:%s}" % (json.dumps(e.src), json.dumps(e.dst), json.dumps(e.prov), e.conf)
                                 for e in es[i:i + batch]) + "]"
            q.append(f"SELECT * FROM cypher('{graph}', $$ UNWIND {lit} AS r MATCH (a {{id:r.s}}), (b {{id:r.d}}) "
                     f"CREATE (a)-[:{rel} {{prov:r.p, conf:r.c}}]->(b) $$) AS (a agtype);")
    return q


# ---------------------------------------------------------------- read side (agent tools)
class KG:
    def __init__(self, nodes: dict, edges: list[Edge], run_time: dict | None = None):
        self.nodes, self.edges = nodes, edges
        self.out, self.inn = defaultdict(list), defaultdict(list)
        for e in edges:
            self.out[e.src].append(e); self.inn[e.dst].append(e)
        self.run_time = run_time or {}

    def _links(self, bug, cutoff=None, hide_run=None):
        """Runs linked to a bug, as visible at `cutoff`: the run being debugged and later links are hidden."""
        return [e.src for e in self.inn[bug] if e.rel == "LINKED_TO" and e.src != hide_run and
                (cutoff is None or self.run_time.get(e.src, cutoff) < cutoff)]

    def domain_view(self, domain: str, cutoff=None, hide_run=None) -> list[tuple[str, int]]:
        d = nid("Domain", domain)
        bugs = [e.src for e in self.inn[d] if e.rel == "BUG_IN_DOMAIN"]
        return sorted(((b, len(self._links(b, cutoff, hide_run))) for b in bugs), key=lambda x: -x[1])

    def bug_view(self, bug: str, cutoff=None, hide_run=None) -> dict:
        b = bug if ":" in bug else nid("Bug", bug)
        runs = self._links(b, cutoff, hide_run)
        tests = Counter(e.dst for r in runs for e in self.out[r] if e.rel == "OF_TEST")
        hosts = Counter(e.dst for r in runs for e in self.out[r] if e.rel == "RAN_ON")
        rel = [e.dst for e in self.out[b] if e.rel == "RELATED_TO"]
        return {"runs": len(runs), "tests": tests.most_common(5), "hosts": hosts.most_common(5), "related": rel,
                "domain": [(e.dst, e.conf) for e in self.out[b] if e.rel == "BUG_IN_DOMAIN"]}

    def expert(self, run: dict, kb, k: int = 5, cutoff=None) -> list[str]:
        """The graph picks the bugs of the run's domain, the KB ranks their text against the run's lines."""
        pool = {b.split(":", 1)[1] for b, _ in self.domain_view(run["domain"], cutoff, nid("Run", run["run"]))}
        hits = kb.search(" ".join(run["lines"]), exclude_run=run["run"], k=200, cutoff=cutoff)
        return [h["bug"] for h in hits if h["bug"] in pool][:k]

    def ppr(self, seeds: dict[str, float], damping: float = .5, iters: int = 30, cutoff=None, hide_run=None) -> dict[str, float]:
        """HippoRAG-style personalised PageRank from the run's entities (test, host, config, ...) over the
        undirected graph; mass that lands on Bug nodes ranks them. Redundant with the classifier's hand-built
        graph features (neutral as a feature), useful as an independent recall channel.
        Leak guard: the run being debugged and every link made at/after `cutoff` are invisible to the walk
        (without it, recall@5 is ~99%: the walk simply follows the run's own link)."""
        adj = defaultdict(list)
        for e in self.edges:
            if e.rel == "LINKED_TO" and (e.src == hide_run or (cutoff is not None and self.run_time.get(e.src, cutoff) >= cutoff)):
                continue
            adj[e.src].append((e.dst, e.conf)); adj[e.dst].append((e.src, e.conf))
        z = sum(seeds.values()) or 1
        reset = {n: v / z for n, v in seeds.items()}
        p = dict(reset)
        for _ in range(iters):
            nxt = defaultdict(float)
            for n, m in p.items():
                nb = adj.get(n, [])
                tot = sum(w for _, w in nb) or 1
                for x, w in nb:
                    nxt[x] += damping * m * w / tot
            for n, v in reset.items():
                nxt[n] += (1 - damping) * v
            p = nxt
        return {n.split(":", 1)[1]: v for n, v in sorted(p.items(), key=lambda x: -x[1]) if n.startswith("Bug:")}

## 1. Synthetic bug reports and discussion threads

In [ ]:
import random
from datetime import timedelta
from synthetic import make_world
from kb_store import Doc, KnowledgeBase
bugs, runs = make_world(n_bugs=60, n_runs=1000, seed=3)
rng = random.Random(0)
for r in runs: r["host"] = f"node{rng.randrange(20):02d}"; r["test"] = f"{r['domain']}/{r['family']}"
docs = []
for b in bugs:
    docs.append(Doc("report", b["id"], None, b["created"], f"<p>{b['title']}</p> Symptom: {'. '.join(b['signature'])}. "
                    f"Root cause: {b['root_cause']} in {b['component']}_driver_{b['domain']}. See https://tracker/x/{b['id']}"))
    for r in [r for r in runs if r["label"] == b["id"]][:3]:
        docs.append(Doc("thread_comment", b["id"], r["run"], r["start"] + timedelta(days=1), f"Seen again on {r['run']}: " + "; ".join(r["lines"][:3])))
    docs.append(Doc("thread_comment", b["id"], None, b["created"] + timedelta(days=2), "+1"))
kb = KnowledgeBase()
print(kb.ingest(docs)); print("re-ingest:", kb.ingest(docs)); print(kb.stats())

## 2. Search: time cut + own-run exclusion, then hybrid

In [ ]:
lab = [r for r in runs if r["label"] != "NONE"]
def recall(kb, k, mode="fuse"):
    return round(sum(r["label"] in [h["bug"] for h in kb.search(" ".join(r["lines"]), exclude_run=r["run"], k=k, mode=mode, cutoff=r["start"])]
                     for r in lab) / len(lab), 3)
print({k: recall(kb, k, "bm25") for k in (1, 5)}, "BM25")

In [ ]:
from sentence_transformers import SentenceTransformer
st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
kb.embed = lambda xs: st.encode(xs, normalize_embeddings=True); print("embedded", kb.embed_new())
print({k: recall(kb, k, "fuse") for k in (1, 5)}, "BM25 + dense (RRF)")

## 3. Knowledge graph: build, purify report, incremental reload, openCypher projection

In [ ]:
from kg import extract, purify, load, project_cypher, KG
nodes, edges = extract(runs, bugs, kb)
nodes, edges, report = purify(nodes, edges)
store = {}
print(report); print("first load", load(store, nodes, edges), "| second load", load(store, nodes, edges))
print(len(nodes), "nodes", len(edges), "edges |", len(project_cypher(nodes, edges)), "Cypher statements"); print(project_cypher(nodes, edges)[2][:200], "...")

## 4. Read side: bug and domain views, graph-scoped expert search, leak-guarded PPR

In [ ]:
g = KG(nodes, edges, {f"Run:{r['run']}": r["start"] for r in runs})
r = max(lab, key=lambda x: x["start"])
print("truth", r["label"]); print("bug view", g.bug_view(r["label"], cutoff=r["start"], hide_run=f"Run:{r['run']}"))
print("expert", g.expert(r, kb, cutoff=r["start"]))
hit = sum(r["label"] in g.expert(r, kb, cutoff=r["start"]) for r in lab) / len(lab)
ppr = lambda r, **kw: list(g.ppr({f"Test:{r['test']}": 1, f"Host:{r['host']}": .5}, **kw))[:5]
leaky = sum(r["label"] in ppr(r) for r in lab[:200]) / 200
clean = sum(r["label"] in ppr(r, cutoff=r["start"], hide_run=f"Run:{r['run']}") for r in lab[:200]) / 200
print(f"expert top-5 {hit:.3f} | PPR recall@5 leaky {leaky:.3f} vs leak-free {clean:.3f}")